# GastroNet — NB3: hybrid_concat baseline (EfficientNet-B4 + ViT-Small, concat fusion)

**Prerequisite**: NB0 (with v2 split) already run on this account.

**What this is, honestly**: this is the concat-fusion **ablation baseline**,
not the project's proposed contribution. Its job is to give an honest number
for the simplest possible fusion approach, so that `hybrid_crossattn`
(NB4/NB5) can later be judged against it fairly. Given how tightly
`cnn_only_v2` and `vit_only_v2` are expected to cluster around 97-98.5% (see
the project handoff doc, Sections 1/2/9), this model beating or losing to
the paper's 98.25% headline is not the point -- what matters is whether
cross-attention fusion later adds anything *over this baseline*.

**Architecture** (per the locked Option C decision from NB2):
- CNN branch: EfficientNet-B4, pretrained, classifier head removed, native
  input **448x448**, produces a 1792-dim feature vector.
- ViT branch: ViT-Small (`timm`, `vit_small_patch16_224`), pretrained,
  classification head removed, native input **224x224**, produces a
  384-dim feature vector.
- Fusion: the two feature vectors are concatenated (1792+384=2176-dim) and
  passed through a small classifier head (`Linear(2176,256) -> ReLU ->
  Dropout(0.3) -> Linear(256,4)`).
- Each image is resized to BOTH 448 and 224 per sample (two tensors per
  sample, one per branch) -- neither branch's native resolution is
  compromised. Augmentation decisions (flip, rotation, color jitter) are
  applied once per sample and shared across both resized copies, so both
  branches see consistent augmented views of the same image rather than
  two independently-randomized versions.

`MODEL_FAMILY = "hybrid_concat_v2"`. Same multi-seed (42, 123, 7), early
stopping (`PATIENCE=6`), checkpointing, and manifest-logging pattern as NB1
and NB2.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, sys, json

ACCOUNT_TAG      = "acct_A"
EXPERIMENTS_ROOT = "/content/drive/MyDrive/gastronet_experiments"
NOTEBOOK_NAME    = "NB3_hybrid_concat_baseline"
RAW_DATASET_DIR  = "/content/drive/MyDrive/gastronet_raw_dataset"

MODEL_FAMILY = "hybrid_concat_v2"
SEEDS_TO_RUN = [42, 123, 7]

CLASS_NAMES = ["Diverticulosis", "Neoplasm", "Peritonitis", "Ureters"]
IMG_SIZE_CNN = 448      # EfficientNet-B4's native training size (matches NB1)
IMG_SIZE_VIT = 224      # ViT-Small's native pretrained size (matches NB2)
BATCH_SIZE = 12         # lower than NB1/NB2's 16 -- this model holds two backbones + two
                         # image tensors per sample in memory at once; drop to 8 if you hit OOM
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 6

SPLIT_JSON_PATH    = os.path.join(EXPERIMENTS_ROOT, "dataset_split_v2.json")
MANIFEST_JSON_PATH = os.path.join(EXPERIMENTS_ROOT, "experiments_manifest.json")

assert os.path.exists(SPLIT_JSON_PATH), (
    "dataset_split_v2.json not found. Run NB0's duplicate-check/v2-split cells first."
)
assert os.path.exists(os.path.join(EXPERIMENTS_ROOT, "checkpoint_utils.py")), (
    "checkpoint_utils.py not found in EXPERIMENTS_ROOT."
)

sys.path.insert(0, EXPERIMENTS_ROOT)
import checkpoint_utils as cku
print("checkpoint_utils imported OK from:", EXPERIMENTS_ROOT)


### Local dataset copy (same reasoning as NB1/NB2 -- avoids Drive rate-limiting)

In [ ]:
import shutil

LOCAL_DATASET_DIR = "/content/gastro_local_copy"

if not os.path.exists(LOCAL_DATASET_DIR):
    os.makedirs(LOCAL_DATASET_DIR)
    for cls in CLASS_NAMES:
        src_dir = os.path.join(RAW_DATASET_DIR, cls)
        dst_dir = os.path.join(LOCAL_DATASET_DIR, cls)
        os.makedirs(dst_dir, exist_ok=True)
        files = os.listdir(src_dir)
        print(f"Copying class '{cls}': {len(files)} files")
        for i, fname in enumerate(files):
            shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))
            if (i + 1) % 200 == 0:
                print(f"  [{cls}] copied {i+1}/{len(files)}")
    print("Local copy complete.")
else:
    print("Local copy already exists this session, skipping copy.")


In [ ]:
with open(SPLIT_JSON_PATH) as f:
    split_payload = json.load(f)

SPLIT_HASH = split_payload["split_hash"]
split = split_payload["split"]
assert split_payload["class_names"] == CLASS_NAMES, "Class name mismatch with locked split!"

print("Loaded split_hash (v2):", SPLIT_HASH)
for k in ["train", "val", "test"]:
    print(f"  {k}: {len(split[k])} images")

def remap_to_local(entries):
    return [(p.replace(RAW_DATASET_DIR, LOCAL_DATASET_DIR), cls) for p, cls in entries]

split = {k: remap_to_local(v) for k, v in split.items()}
print("Paths remapped to local copy, e.g.:", split["train"][0][0])


In [ ]:
import torch
import random
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

_to_tensor_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class GastroDualDataset(Dataset):
    '''
    Returns (cnn_tensor, vit_tensor, label) per sample -- one image resized
    two ways, one per branch, per the locked Option C decision (no branch's
    native resolution is compromised). Augmentation decisions (flip,
    rotation, color jitter) are drawn once per sample and applied to BOTH
    resized copies, so the two branches see consistent augmented views of
    the same underlying image rather than two independently-randomized ones.
    '''
    def __init__(self, entries, train_mode):
        self.entries = entries
        self.train_mode = train_mode

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        path, cls = self.entries[idx]
        img = Image.open(path).convert("RGB")

        if self.train_mode:
            if random.random() < 0.5:
                img = TF.hflip(img)
            angle = random.uniform(-10, 10)
            img = TF.rotate(img, angle)
            brightness = random.uniform(0.9, 1.1)
            contrast = random.uniform(0.9, 1.1)
            img = TF.adjust_brightness(img, brightness)
            img = TF.adjust_contrast(img, contrast)

        cnn_img = img.resize((IMG_SIZE_CNN, IMG_SIZE_CNN))
        vit_img = img.resize((IMG_SIZE_VIT, IMG_SIZE_VIT))

        cnn_tensor = _to_tensor_norm(cnn_img)
        vit_tensor = _to_tensor_norm(vit_img)
        label = class_to_idx[cls]
        return cnn_tensor, vit_tensor, label


train_ds = GastroDualDataset(split["train"], train_mode=True)
val_ds   = GastroDualDataset(split["val"], train_mode=False)
test_ds  = GastroDualDataset(split["test"], train_mode=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} batches/epoch={len(train_loader)}")


In [ ]:
!pip install timm --break-system-packages -q

import timm
import torch.nn as nn
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights


class HybridConcatModel(nn.Module):
    def __init__(self, num_classes=len(CLASS_NAMES)):
        super().__init__()

        # CNN branch: EfficientNet-B4, classifier removed, keep feature extractor + pooling.
        cnn_weights = EfficientNet_B4_Weights.IMAGENET1K_V1
        cnn = efficientnet_b4(weights=cnn_weights)
        self.cnn_features = cnn.features
        self.cnn_pool = cnn.avgpool
        cnn_feature_dim = cnn.classifier[1].in_features   # 1792 for EfficientNet-B4

        # ViT branch: ViT-Small, classification head removed via num_classes=0
        # (timm returns the pooled feature vector directly in this mode).
        self.vit = timm.create_model("vit_small_patch16_224", pretrained=True, num_classes=0)
        vit_feature_dim = self.vit.num_features            # 384 for vit_small

        fused_dim = cnn_feature_dim + vit_feature_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

        self.cnn_feature_dim = cnn_feature_dim
        self.vit_feature_dim = vit_feature_dim

    def forward(self, cnn_img, vit_img):
        cnn_feat = self.cnn_features(cnn_img)
        cnn_feat = self.cnn_pool(cnn_feat)
        cnn_feat = torch.flatten(cnn_feat, 1)          # (B, 1792)

        vit_feat = self.vit(vit_img)                    # (B, 384)

        fused = torch.cat([cnn_feat, vit_feat], dim=1)  # (B, 2176)
        return self.classifier(fused)


def build_hybrid_concat_model(num_classes=len(CLASS_NAMES)):
    return HybridConcatModel(num_classes)

_probe = build_hybrid_concat_model()
print(f"CNN branch feature dim: {_probe.cnn_feature_dim}")
print(f"ViT branch feature dim: {_probe.vit_feature_dim}")
print(f"Total params: {sum(p.numel() for p in _probe.parameters()):,}")
del _probe


In [ ]:
import time

def run_epoch(model, optimizer, scaler, criterion, loader, train_mode, print_every=20):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    start_time = time.time()
    last_print_time = start_time

    with torch.set_grad_enabled(train_mode):
        for batch_idx, (cnn_imgs, vit_imgs, labels) in enumerate(loader):
            cnn_imgs = cnn_imgs.to(device, non_blocking=True)
            vit_imgs = vit_imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if train_mode:
                optimizer.zero_grad()

            with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
                outputs = model(cnn_imgs, vit_imgs)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * cnn_imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += cnn_imgs.size(0)

            if (batch_idx + 1) % print_every == 0:
                now = time.time()
                chunk_time = now - last_print_time
                last_print_time = now
                running_acc = correct / total
                mode_str = "train" if train_mode else "val"
                print(f"    [{mode_str}] batch {batch_idx+1}/{len(loader)} "
                      f"| running_acc={running_acc:.4f} "
                      f"| this_chunk={chunk_time:.1f}s | total_elapsed={now - start_time:.1f}s")

    return total_loss / total, correct / total


In [ ]:
all_seed_results = {}

for SEED in SEEDS_TO_RUN:
    seed_exp_dir = cku.get_experiment_dir(EXPERIMENTS_ROOT, MODEL_FAMILY, SEED)

    if os.path.exists(os.path.join(seed_exp_dir, "results.json")):
        print(f"Seed {SEED} already has results.json, skipping.")
        with open(os.path.join(seed_exp_dir, "results.json")) as f:
            all_seed_results[SEED] = json.load(f)
        continue

    print(f"\n{'='*60}\nStarting SEED={SEED}\n{'='*60}")

    torch.manual_seed(SEED)
    model = build_hybrid_concat_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    criterion = nn.CrossEntropyLoss()

    start_epoch, best_val_acc, history = cku.resume_or_start(
        seed_exp_dir, model, optimizer, scheduler, scaler, map_location=device
    )
    cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                       status="resumed" if start_epoch > 0 else "started",
                       best_val_acc=best_val_acc, drive_path=seed_exp_dir)

    epochs_without_improvement = 0

    try:
        for epoch in range(start_epoch, NUM_EPOCHS):
            train_loss, train_acc = run_epoch(model, optimizer, scaler, criterion, train_loader, train_mode=True)
            val_loss, val_acc = run_epoch(model, optimizer, scaler, criterion, val_loader, train_mode=False)
            scheduler.step()

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            print(f"[seed {SEED}] Epoch {epoch+1}/{NUM_EPOCHS} | "
                  f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

            cku.save_latest(seed_exp_dir, epoch, model, optimizer, scheduler, scaler, best_val_acc, history)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                epochs_without_improvement = 0
                config = {
                    "model_family": MODEL_FAMILY, "seed": SEED, "split_hash": SPLIT_HASH,
                    "account_tag": ACCOUNT_TAG, "notebook_name": NOTEBOOK_NAME,
                    "img_size_cnn": IMG_SIZE_CNN, "img_size_vit": IMG_SIZE_VIT,
                    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
                    "class_names": CLASS_NAMES, "fusion_type": "concat",
                }
                cku.save_best(seed_exp_dir, model, epoch, best_val_acc, config)
                cku.save_config(seed_exp_dir, config)
                print(f"  -> new best_val_acc={best_val_acc:.4f}, saved best.pt")
            else:
                epochs_without_improvement += 1
                print(f"  -> no improvement for {epochs_without_improvement}/{PATIENCE} epochs")
                if epochs_without_improvement >= PATIENCE:
                    print(f"Stopping early for seed {SEED}: no improvement for {PATIENCE} epochs.")
                    cku.save_history(seed_exp_dir, history)
                    break

            cku.save_history(seed_exp_dir, history)

    except KeyboardInterrupt:
        cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                           status="interrupted", best_val_acc=best_val_acc, drive_path=seed_exp_dir,
                           note="Manually interrupted -- latest.pt has current state, safe to resume.")
        print(f"Interrupted during seed {SEED}. Re-run this cell to resume from the last completed epoch.")
        break

    # Test-set evaluation for this seed
    best_ckpt = cku.load_best(seed_exp_dir, map_location=device)
    cku.assert_split_hash_matches(best_ckpt["config"], SPLIT_HASH)
    eval_model = build_hybrid_concat_model().to(device)
    eval_model.load_state_dict(best_ckpt["model_state_dict"])
    eval_model.eval()

    correct, total, preds_all, labels_all = 0, 0, [], []
    with torch.no_grad():
        for cnn_imgs, vit_imgs, labels in test_loader:
            cnn_imgs, vit_imgs, labels = cnn_imgs.to(device), vit_imgs.to(device), labels.to(device)
            preds = eval_model(cnn_imgs, vit_imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += cnn_imgs.size(0)
            preds_all.extend(preds.cpu().tolist())
            labels_all.extend(labels.cpu().tolist())

    test_acc = correct / total
    results = {
        "model_family": MODEL_FAMILY, "seed": SEED, "split_hash": SPLIT_HASH,
        "account_tag": ACCOUNT_TAG, "best_epoch": best_ckpt["epoch"],
        "best_val_acc": best_ckpt["best_val_acc"], "test_accuracy": test_acc,
        "predictions": preds_all, "labels": labels_all, "class_names": CLASS_NAMES,
    }
    cku.save_results(seed_exp_dir, results)
    all_seed_results[SEED] = results

    cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                       status="completed", best_val_acc=best_val_acc, drive_path=seed_exp_dir)

    cku.finalize_experiment(seed_exp_dir)
    print(f"Seed {SEED} done. test_acc={test_acc:.4f}. latest.pt cleaned up.\n")

print("\nSeeds completed this run:")
for s, r in all_seed_results.items():
    print(f"  seed {s}: test_accuracy={r['test_accuracy']:.4f}, best_val_acc={r['best_val_acc']:.4f}")


In [ ]:
cku.manifest_summary(MANIFEST_JSON_PATH)


In [ ]:
import numpy as np
import glob

def load_seed_results(model_family):
    results = []
    for rf in sorted(glob.glob(os.path.join(EXPERIMENTS_ROOT, model_family, "seed_*", "results.json"))):
        with open(rf) as f:
            results.append(json.load(f))
    return results

for family in ["cnn_only_v2", "vit_only_v2", MODEL_FAMILY]:
    results = load_seed_results(family)
    if not results:
        print(f"{family}: no results found yet")
        continue
    accs = [r["test_accuracy"] for r in results]
    seeds = [r["seed"] for r in results]
    if len(accs) > 1:
        print(f"{family:20s} seeds={seeds} mean={np.mean(accs):.4f} std={np.std(accs, ddof=1):.4f}")
    else:
        print(f"{family:20s} seeds={seeds} (only one seed so far) acc={accs[0]:.4f}")


## After this notebook finishes

1. **Memory**: `BATCH_SIZE=12` here (vs 16 in NB1/NB2) because this model
   holds two backbones and two image tensors per sample simultaneously. If
   you hit CUDA OOM, drop to 8.
2. **Expected outcome, honestly**: this model may land close to, above, or
   below both `cnn_only_v2` and `vit_only_v2` -- concatenation fusion isn't
   expected to reliably beat a strong single backbone on a near-saturated
   dataset. Don't be surprised either way. What matters is having this
   number as the reference point for `hybrid_crossattn` later.
3. **Overfitting**: watch the same train/val gap pattern seen in NB1 --
   early stopping (`PATIENCE=6`) should catch it, but glance at
   `history.json` per seed regardless.
4. **Comparison cell** (second to last): prints mean±std for `cnn_only_v2`,
   `vit_only_v2`, and `hybrid_concat_v2` side by side, so you can see where
   this baseline actually lands relative to the single-branch models the
   moment all three have results.
5. **Next: NB4** -- `hybrid_crossattn` model definition + a dummy-tensor
   shape/forward-pass sanity check (no real training yet), implementing the
   architectural decisions from the project handoff doc Section 2:
   unidirectional cross-attention (CNN tokens as queries attending to
   ViT-Small keys/values), learned 2D positional embedding on CNN spatial
   tokens, `d_model=256` fusion block, content-only queries. This notebook's
   dual-resolution dataset pattern (`GastroDualDataset`) carries over
   directly into NB4/NB5's data pipeline.
